# YOLO Dataset Preparation

In the previous notebook, the Rico UI annotations were converted into YOLO label files.

This notebook prepares the dataset for model training by:

1. Verifying images and labels.
2. Creating the YOLO dataset structure.
3. Splitting the dataset into train, validation, and test sets.
4. Copying images and labels.
5. Creating the `data.yaml` configuration file.

In [3]:
import os
import glob
import shutil
import random

from pathlib import Path

In [4]:
image_folder = "../Data/semantic_annotations"

label_folder = "../labels"

dataset_folder = "../dataset"

In [5]:
image_files = sorted(glob.glob(os.path.join(image_folder, "*.png")))

print("Images:", len(image_files))

Images: 66261


In [6]:
label_files = sorted(glob.glob(os.path.join(label_folder, "*.txt")))

print("Labels:", len(label_files))

Labels: 66261


In [7]:
assert len(image_files) == len(label_files)

print("Dataset verified successfully.")

Dataset verified successfully.


## Creating the YOLO Dataset Structure

YOLO expects the dataset to be organized into separate folders for training,
validation, and testing images and labels.

The following directory structure will be created automatically.

In [8]:
folders = [
    "images/train",
    "images/val",
    "images/test",
    "labels/train",
    "labels/val",
    "labels/test",
]

for folder in folders:
    Path(os.path.join(dataset_folder, folder)).mkdir(parents=True, exist_ok=True)

print("YOLO folder structure created successfully.")

YOLO folder structure created successfully.


## Randomizing the Dataset

The dataset is shuffled before splitting to ensure that the training,
validation, and testing sets contain a representative distribution of UI screens.

In [10]:
random.seed(42)

random.shuffle(image_files)

print("Dataset shuffled successfully.")

Dataset shuffled successfully.


In [11]:
total_images = len(image_files)

train_size = int(total_images * 0.8)
val_size = int(total_images * 0.1)

train_images = image_files[:train_size]

val_images = image_files[train_size : train_size + val_size]

test_images = image_files[train_size + val_size :]

In [12]:
print("Training :", len(train_images))
print("Validation :", len(val_images))
print("Testing :", len(test_images))
print()
print("Total :", len(train_images) + len(val_images) + len(test_images))

Training : 53008
Validation : 6626
Testing : 6627

Total : 66261


## Copying Images and Labels

Each image has a corresponding YOLO annotation file.

For every dataset split (train, validation and test):

- Copy the image into the images folder.
- Copy the matching label into the labels folder.

This prepares the dataset in the format required by YOLO.

In [15]:
import shutil
from pathlib import Path

In [18]:
def copy_dataset(image_list, split_name):

    for image_path in image_list:

        file_name = Path(image_path).stem

        label_path = os.path.join(label_folder, file_name + ".txt")

        image_destination = os.path.join(
            dataset_folder, "images", split_name, file_name + ".png"
        )

        label_destination = os.path.join(
            dataset_folder, "labels", split_name, file_name + ".txt"
        )

        shutil.copy(image_path, image_destination)
        shutil.copy(label_path, label_destination)

In [19]:
copy_dataset(train_images, "train")

print("Training data copied.")

Training data copied.


In [20]:
copy_dataset(val_images, "val")

print("Validation data copied.")

Validation data copied.


In [21]:
copy_dataset(test_images, "test")

print("Testing data copied.")

Testing data copied.


In [22]:
print(
    "Train Images :", len(os.listdir(os.path.join(dataset_folder, "images", "train")))
)
print(
    "Train Labels :", len(os.listdir(os.path.join(dataset_folder, "labels", "train")))
)

print()

print(
    "Validation Images :",
    len(os.listdir(os.path.join(dataset_folder, "images", "val"))),
)
print(
    "Validation Labels :",
    len(os.listdir(os.path.join(dataset_folder, "labels", "val"))),
)

print()

print("Test Images :", len(os.listdir(os.path.join(dataset_folder, "images", "test"))))
print("Test Labels :", len(os.listdir(os.path.join(dataset_folder, "labels", "test"))))

Train Images : 53008
Train Labels : 53008

Validation Images : 6626
Validation Labels : 6626

Test Images : 6627
Test Labels : 6627


## Creating the YOLO Configuration File

YOLO uses a `data.yaml` file to locate the dataset and understand the object classes.

This file specifies:

- dataset location
- training images
- validation images
- testing images
- class names

In [24]:
CLASS_MAP = {
    "Button": 0,
    "CheckBox": 1,
    "CheckedTextView": 2,
    "EditText": 3,
    "ImageButton": 4,
    "ImageView": 5,
    "ProgressBar": 6,
    "RadioButton": 7,
    "RatingBar": 8,
    "SeekBar": 9,
    "Spinner": 10,
    "Switch": 11,
    "SwitchCompat": 12,
    "TextView": 13,
    "ToggleButton": 14,
    "VideoView": 15,
    "AutoCompleteTextView": 16,
    "MultiAutoCompleteTextView": 17,
    "Chronometer": 18,
    "DigitalClock": 19,
    "AnalogClock": 20,
    "DatePicker": 21,
    "TimePicker": 22,
    "WebView": 23,
    "MapView": 24,
}
class_names = list(CLASS_MAP.keys())

class_names

['Button',
 'CheckBox',
 'CheckedTextView',
 'EditText',
 'ImageButton',
 'ImageView',
 'ProgressBar',
 'RadioButton',
 'RatingBar',
 'SeekBar',
 'Spinner',
 'Switch',
 'SwitchCompat',
 'TextView',
 'ToggleButton',
 'VideoView',
 'AutoCompleteTextView',
 'MultiAutoCompleteTextView',
 'Chronometer',
 'DigitalClock',
 'AnalogClock',
 'DatePicker',
 'TimePicker',
 'WebView',
 'MapView']

In [25]:
import yaml

yaml_data = {
    "path": "../dataset",
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": len(class_names),
    "names": class_names,
}

In [26]:
with open("../dataset/data.yaml", "w") as f:
    yaml.dump(yaml_data, f, sort_keys=False)

print("data.yaml created successfully.")

data.yaml created successfully.
